In [2]:
import cv2
import os
import torch
from ultralytics import YOLO
from pathlib import Path
from tqdm import tqdm

# ==========================================
# 1. CONFIGURAZIONE
# ==========================================
# ⚠️ MODIFICA QUI CON IL NOME DEL TUO VIDEO
VIDEO_INPUT_NAME = "test_lyon.mp4" 

# Imposta i percorsi
PROJECT_ROOT = Path(os.getcwd()).parent 
INPUT_DIR = Path(r"C:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\data")        # Dove hai messo il video (es. root del progetto)
OUTPUT_DIR = PROJECT_ROOT / "results" / "demo_videos"  # Dove salvare il video di output
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = INPUT_DIR / VIDEO_INPUT_NAME
OUTPUT_PATH = OUTPUT_DIR / f"demo_{VIDEO_INPUT_NAME}"

# Configurazione Hardware
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"⚡ Dispositivo di calcolo: {DEVICE}")

# ==========================================
# 2. CARICAMENTO MODELLO
# ==========================================
# Cerca il tuo modello 'best.pt'
custom_model_path = PROJECT_ROOT / 'runs/yolov8n_vehicle_detection2/weights/best.pt'

if custom_model_path.exists():
    print(f"✅ Carico il TUO modello addestrato: {custom_model_path}")
    model = YOLO(str(custom_model_path))
else:
    print("⚠️ Modello custom non trovato. Uso YOLOv8n pre-trained standard.")
    model = YOLO("yolov8n.pt")

# ==========================================
# 3. ELABORAZIONE VIDEO
# ==========================================
def process_custom_video():
    if not VIDEO_PATH.exists():
        print(f"❌ Errore: Non trovo il file video: {VIDEO_PATH}")
        print("Assicurati di averlo caricato e che il nome sia corretto.")
        return

    # Apre il video
    cap = cv2.VideoCapture(str(VIDEO_PATH))
    
    # Legge proprietà video originale
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"🎬 Video aperto: {width}x{height} @ {fps}fps ({total_frames} frames)")
    print(f"💾 Salvataggio in: {OUTPUT_PATH}")

    # Setup Video Writer
    # Usiamo 'mp4v' per MP4. Se dà problemi, prova 'MJPG' con estensione .avi
    writer = cv2.VideoWriter(str(OUTPUT_PATH), cv2.VideoWriter_fourcc(*'mp4v'), fps, (width, height))

    # Barra di progresso
    pbar = tqdm(total=total_frames, desc="Processing Video")

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # --- INFERENZA ---
        # conf=0.25 scarta rilevamenti deboli
        # persist=True mantiene gli ID stabili tra i frame
        results = model.track(frame, persist=True, conf=0.25, verbose=False, device=DEVICE)[0]

        # --- DISEGNO ---
        # Il metodo .plot() di YOLO disegna automaticamente box, label e confidenza
        # È il modo più pulito per mostrare che il modello funziona.
        annotated_frame = results.plot()

        # Scrittura frame
        writer.write(annotated_frame)
        pbar.update(1)

    # Rilascio risorse
    cap.release()
    writer.release()
    pbar.close()
    
    print("\n" + "="*50)
    print("✅ ELABORAZIONE COMPLETATA!")
    print(f"📂 Il video demo è pronto: {OUTPUT_PATH}")
    print("="*50)

# Avvia lo script
process_custom_video()

⚡ Dispositivo di calcolo: cuda
✅ Carico il TUO modello addestrato: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\runs\yolov8n_vehicle_detection2\weights\best.pt
🎬 Video aperto: 480x848 @ 29fps (1815 frames)
💾 Salvataggio in: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results\demo_videos\demo_test_lyon.mp4


Processing Video: 100%|██████████| 1815/1815 [01:09<00:00, 26.20it/s]


✅ ELABORAZIONE COMPLETATA!
📂 Il video demo è pronto: c:\Users\alera\Desktop\ENTPE\ICVIS\project\ICV_project\results\demo_videos\demo_test_lyon.mp4
